In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import numpy as np
import pandas as pd
import pickle


In [2]:
## Load the saved model
model = load_model('model.h5')

### Load the saved scaler
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

### Load the saved label encoder
with open('label_encoder_gender.pkl', 'rb') as f:
    label_encoder_gender = pickle.load(f)  

## load the saved one-hot encoder
with open('onehot_encoder_geo.pkl', 'rb') as f:      
    onehot_encoder_geo = pickle.load(f)      


In [3]:
# Example input data for prediction
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [4]:
#One-hot encode the 'Geography' feature
geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

c:\Users\ASUS\OneDrive\Documents\Desktop\ANN Classification\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [5]:
input_data_df = pd.DataFrame([input_data])
input_data_df 

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [6]:
# Encode the 'Gender' feature using the label encoder
input_data_df['Gender'] = label_encoder_gender.transform(input_data_df['Gender'])
input_data_df



,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [7]:
## concatination one -hot encoded 'Geography' features with the rest of the input data
input_data_df = pd.concat([input_data_df.drop('Geography', axis=1), geo_encoded_df], axis=1)
input_data_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [8]:
## Scalling the input data 
input_data_scaled = scaler.transform(input_data_df)
input_data_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [9]:
prediction = model.predict(input_data_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 298ms/step


array([[0.05595936]], dtype=float32)

In [10]:
prediction_probability = prediction[0][0]

In [12]:
if prediction_probability > 0.5:
    print(f"The customer will leave the bank with a probability of {prediction_probability:.2f}.")
else:
    print(f"The customer will not leave the bank with a probability of {prediction_probability:.2f}.")

The customer will not leave the bank with a probability of 0.06.
